# Notebook 37 — Corrected V4.2 Standalone Serving Preprocessing

Export the train-fitted entity/categorical vocabularies, numerical statistics, and request schema required to encode online user state without reading Gold data at serving time. This is a CPU/High-RAM job; no GPU is required.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile
DRIVE_ROOT=Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT=DRIVE_ROOT/'data/gold/adaptive/v1'; BASE_GOLD_ROOT=DRIVE_ROOT/'data/gold/two_tower/v1_colab'
WORK_ROOT=Path('/content/recsys_serving_preprocessing'); SOURCE_ROOT=WORK_ROOT/'runtime'; V4_DIR=WORK_ROOT/'v4'; OUTPUT_DIR=WORK_ROOT/'preprocessing'
DRIVE_REPORT_DIR=DRIVE_ROOT/'artifacts/adaptive/serving_preprocessing_v1'
for p in (SOURCE_ROOT,V4_DIR,OUTPUT_DIR,DRIVE_REPORT_DIR): p.mkdir(parents=True,exist_ok=True)

In [ ]:
required={'torch':'torch','pandas':'pandas','pyarrow':'pyarrow','huggingface_hub':'huggingface_hub>=0.32'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
required_paths=[ADAPTIVE_ROOT/'sequence_examples/train',BASE_GOLD_ROOT/'transforms/numeric_stats.json',BASE_GOLD_ROOT/'vocabularies/user_active_degree',BASE_GOLD_ROOT/'vocabularies/video_type',BASE_GOLD_ROOT/'vocabularies/upload_type']
assert all(p.exists() for p in required_paths),'Required Gold inputs are missing.'
free=shutil.disk_usage('/content').free/1024**3; assert free>=8,f'Need 8 GiB local disk; found {free:.1f}'
print('CPU/Drive/disk preflight: PASS',f'{free:.1f} GiB')

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
REPO='khoa1505/recsys-artifacts'; RUNTIME='runtime/corrected_v4_preprocessing_runtime_v1.zip'; RUNTIME_SHA='0d3e5d292ffdab9d9a7515f740a25659c7328f7a61ae96509f9651e90fba01b6'; V4_FILE='v4_corrected/v1/best_model.pt'
def get_token(name):
    try: raw=userdata.get(name)
    except Exception: raw=None
    if isinstance(raw,dict): raw=next((v for v in raw.values() if isinstance(v,str) and v.startswith('hf_')),None)
    return raw.strip() if isinstance(raw,str) and raw.strip().startswith('hf_') else None
READ=get_token('HF_TOKEN') or getpass('Paste Hugging Face READ token: ').strip(); WRITE=get_token('HF_WRITE_TOKEN') or getpass('Paste Hugging Face WRITE token: ').strip()
assert READ.startswith('hf_') and WRITE.startswith('hf_')
runtime=Path(hf_hub_download(REPO,RUNTIME,repo_type='model',token=READ)); checkpoint=Path(hf_hub_download(REPO,V4_FILE,repo_type='model',token=READ))
assert hashlib.sha256(runtime.read_bytes()).hexdigest()==RUNTIME_SHA and checkpoint.stat().st_size>100_000_000
target=V4_DIR/'best_model.pt'; target.unlink(missing_ok=True); target.symlink_to(checkpoint.resolve())
shutil.rmtree(SOURCE_ROOT,ignore_errors=True); SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime) as z: assert z.testzip() is None; z.extractall(SOURCE_ROOT)
SCRIPT=SOURCE_ROOT/'scripts/export_corrected_v4_serving_preprocessing.py'; assert SCRIPT.exists(); compile(SCRIPT.read_text(),str(SCRIPT),'exec')
print('runtime/checkpoint integrity: PASS')

In [ ]:
shutil.rmtree(OUTPUT_DIR,ignore_errors=True); OUTPUT_DIR.mkdir(parents=True)
command=[sys.executable,str(SCRIPT),'--adaptive-root',str(ADAPTIVE_ROOT),'--base-gold-root',str(BASE_GOLD_ROOT),'--corrected-v4-dir',str(V4_DIR),'--output-dir',str(OUTPUT_DIR)]
log_path=Path('/content/serving_preprocessing.log'); process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
with log_path.open('w') as log:
    for line in process.stdout: print(line,end=''); log.write(line); log.flush()
code=process.wait()
if code!=0: raise RuntimeError('Export failed:\n'+'\n'.join(log_path.read_text(errors='replace').splitlines()[-200:]))
manifest=json.loads((OUTPUT_DIR/'manifest.json').read_text())
assert manifest['gold_data_required_at_runtime'] is False and manifest['standalone_encoder_smoke']['passed'] is True
print('standalone preprocessing export: PASS',manifest['vocab_sizes'])

In [ ]:
api=HfApi(token=WRITE); remote='serving/corrected_v4_2_preprocessing_v1'
api.upload_folder(repo_id=REPO,repo_type='model',folder_path=str(OUTPUT_DIR),path_in_repo=remote,commit_message='Upload Corrected V4.2 serving preprocessing')
for name in [*manifest['files'],'manifest.json']: assert api.file_exists(REPO,f'{remote}/{name}',repo_type='model')
shutil.copy2(OUTPUT_DIR/'manifest.json',DRIVE_REPORT_DIR/'manifest.json'); shutil.copy2(log_path,DRIVE_REPORT_DIR/log_path.name)
small=WORK_ROOT/'report'; shutil.rmtree(small,ignore_errors=True); small.mkdir(); shutil.copy2(OUTPUT_DIR/'manifest.json',small/'manifest.json'); shutil.copy2(log_path,small/log_path.name)
archive=Path(shutil.make_archive('/content/corrected_v4_serving_preprocessing_report','zip',small))
print('Hugging Face verification: PASS'); files.download(str(archive))